# LeetCode 1984 — Minimum Difference Between Highest and Lowest of K Scores bằng Python

**Link đề:** https://leetcode.com/problems/minimum-difference-between-highest-and-lowest-of-k-scores/
**Độ khó:** Easy · **Chủ đề:** Array, Sorting, Sliding Window

```python
class Solution:
    def minimumDifference(self, nums: List[int], k: int) -> int:

```

## Mục tiêu học
1. Hiểu đề: chọn **k** học sinh sao cho (điểm cao nhất − điểm thấp nhất) trong nhóm là **nhỏ nhất**.
2. Biết cách vét cạn bằng `itertools.combinations` và thấy vì sao nó bùng nổ.
3. Hiểu insight quan trọng: **sau khi sắp xếp, nhóm tốt nhất luôn là k phần tử liền kề**.
4. Viết lời giải **sort + cửa sổ trượt (sliding window) kích thước cố định**, trace từng cửa sổ.
5. Chứng minh đúng bằng lập luận "thay thế", phân tích `O(n log n)`, test đối chiếu, làm bài tập biến thể.

## 1. Đề bài yêu cầu gì?

Mảng `nums` (đánh chỉ số từ 0), `nums[i]` là điểm của học sinh thứ `i`. Cho số nguyên `k`.

Chọn điểm của **bất kỳ k học sinh** sao cho **hiệu giữa điểm cao nhất và thấp nhất** trong k điểm đó **nhỏ nhất**. Trả về hiệu nhỏ nhất đó.

| `nums` | `k` | Kết quả | Giải thích |
|---|:-:|:-:|---|
| `[90]` | 1 | `0` | chỉ 1 người: 90 − 90 = 0 |
| `[9,4,1,7]` | 2 | `2` | chọn {9, 7}: 9 − 7 = 2 |
| `[1,4,7,9]` (đã sort) | 3 | `5` | chọn {4, 7, 9}: 9 − 4 = 5 |

**Constraints:** `1 <= k <= nums.length <= 1000`, `0 <= nums[i] <= 10^5`.

Chú ý: "chọn bất kỳ" → **không** cần liền kề trong mảng gốc. Thứ tự gốc không quan trọng.

In [1]:
# Liệt kê tay 6 cách chọn 2 người trong ví dụ 2
nums = [9, 4, 1, 7]
for i in range(len(nums)):
    for j in range(i + 1, len(nums)):
        a, c = nums[i], nums[j]
        print(f"chọn ({a}, {c}) -> hiệu = {max(a, c) - min(a, c)}")

chọn (9, 4) -> hiệu = 5
chọn (9, 1) -> hiệu = 8
chọn (9, 7) -> hiệu = 2
chọn (4, 1) -> hiệu = 3
chọn (4, 7) -> hiệu = 3
chọn (1, 7) -> hiệu = 6


## 2. Nền tảng Python cần biết

### 2.1. `sorted()` và `list.sort()`

- `sorted(nums)` tạo **list mới** đã sắp xếp tăng dần, list gốc giữ nguyên.
- `nums.sort()` sắp xếp **tại chỗ** (in-place), trả về `None`.
- Độ phức tạp: `O(n log n)` (thuật toán Timsort).

In [2]:
nums = [9, 4, 1, 7]
s = sorted(nums)
print("sorted:", s, "| gốc vẫn:", nums)

nums2 = [9, 4, 1, 7]
r = nums2.sort()
print("sau .sort():", nums2, "| giá trị trả về:", r)

sorted: [1, 4, 7, 9] | gốc vẫn: [9, 4, 1, 7]
sau .sort(): [1, 4, 7, 9] | giá trị trả về: None


### 2.2. Chỉ số và slicing `a[i:i+k]`

`a[i:i+k]` lấy **k phần tử liên tiếp** bắt đầu từ chỉ số `i` (không lấy `i+k`). Với `a` đã sort, phần tử nhỏ nhất của đoạn là `a[i]`, lớn nhất là `a[i+k-1]`.

In [3]:
a = [1, 4, 7, 9]
k = 3
for i in range(len(a) - k + 1):
    window = a[i:i + k]
    print(f"i={i}: a[{i}:{i+k}] = {window} -> đầu {a[i]}, cuối a[{i+k-1}] = {a[i+k-1]}")

i=0: a[0:3] = [1, 4, 7] -> đầu 1, cuối a[2] = 7
i=1: a[1:4] = [4, 7, 9] -> đầu 4, cuối a[3] = 9


### 2.3. Vì sao `range(len(a) - k + 1)`?

Chỉ số bắt đầu cửa sổ cuối cùng là `n - k` (để `n - k + k - 1 = n - 1` là phần tử cuối). Vì `range` không lấy `stop`, ta viết `range(n - k + 1)` để có `0, 1, ..., n - k`.

- `n = 4, k = 3` → `i ∈ {0, 1}` → 2 cửa sổ.
- `n = 4, k = 4` → `i ∈ {0}` → 1 cửa sổ.
- `n = 1, k = 1` → `i ∈ {0}`.

Tổng quát: có **`n − k + 1` cửa sổ**.

In [4]:
for n, k in [(4, 3), (4, 4), (1, 1), (5, 2)]:
    print(f"n={n}, k={k}: các i = {list(range(n - k + 1))}")

n=4, k=3: các i = [0, 1]
n=4, k=4: các i = [0]
n=1, k=1: các i = [0]
n=5, k=2: các i = [0, 1, 2, 3]


### 2.4. `itertools.combinations` — sinh mọi cách chọn

`combinations(nums, k)` sinh ra mọi bộ `k` phần tử (theo vị trí, không lặp, không quan tâm thứ tự). Số bộ là $\binom{n}{k}$, tăng cực nhanh.

In [5]:
from itertools import combinations
from math import comb

print(list(combinations([9, 4, 1, 7], 2)))
print("C(4,2) =", comb(4, 2))
print("C(1000, 500) có", len(str(comb(1000, 500))), "chữ số!")

[(9, 4), (9, 1), (9, 7), (4, 1), (4, 7), (1, 7)]
C(4,2) = 6
C(1000, 500) có 300 chữ số!


### 2.5. `float('inf')` — giá trị khởi tạo cho bài "tìm nhỏ nhất"

Khi tìm min, ta khởi tạo `best = float('inf')` (dương vô cùng) để bất kỳ giá trị thật nào cũng nhỏ hơn nó.

In [6]:
best = float('inf')
for v in [5, 8, 2, 6]:
    best = min(best, v)
    print("sau", v, "-> best =", best)

sau 5 -> best = 5
sau 8 -> best = 5
sau 2 -> best = 2
sau 6 -> best = 2


## 3. Cách 1 — Vét cạn mọi nhóm k người

Ý tưởng trực giác: thử **mọi** nhóm `k` người, tính `max − min`, giữ giá trị nhỏ nhất.

Đúng, nhưng chỉ dùng được với `n` rất nhỏ.

In [7]:
def min_diff_bruteforce(nums, k):
    best = float('inf')
    for group in combinations(nums, k):
        best = min(best, max(group) - min(group))
    return best

print(min_diff_bruteforce([90], 1))        # 0
print(min_diff_bruteforce([9, 4, 1, 7], 2))  # 2
print(min_diff_bruteforce([9, 4, 1, 7], 3))  # 5

0
2
5


### Nút thắt của vét cạn

- Số nhóm: $\binom{n}{k}$. Với `n = 1000`, `k = 500` → con số có tới 300 chữ số → **không bao giờ chạy xong**.
- Mỗi nhóm còn tốn `O(k)` để tính max/min.

Ta cần trả lời: **có cần thử mọi nhóm không?** Nhóm nào chắc chắn tệ, có thể bỏ qua?

In [8]:
for n, k in [(10, 5), (20, 10), (30, 15), (40, 20)]:
    print(f"n={n:>2}, k={k:>2}: số nhóm = {comb(n, k):,}")

n=10, k= 5: số nhóm = 252
n=20, k=10: số nhóm = 184,756
n=30, k=15: số nhóm = 155,117,520
n=40, k=20: số nhóm = 137,846,528,820


## 4. Insight — Sắp xếp rồi chỉ xét k phần tử liền kề

Quan sát trên mảng đã sort `a = [1, 4, 7, 9]`, `k = 3`:

| Nhóm | max − min |
|---|:-:|
| {1, 4, 7} (liền kề) | 6 |
| {4, 7, 9} (liền kề) | **5** |
| {1, 4, 9} (có lỗ hổng: bỏ 7) | 8 |
| {1, 7, 9} (có lỗ hổng: bỏ 4) | 8 |

Nhóm có "lỗ hổng" luôn tệ hơn hoặc bằng nhóm liền kề cùng đầu/cuối.

**Mấu chốt:** nếu nhóm có min = `a[i]` và max = `a[j]`, thì mọi phần tử giữa `a[i]` và `a[j]` đều nằm trong khoảng `[a[i], a[j]]`. Nhóm `k` phần tử liên tiếp `a[i..i+k-1]` có max `a[i+k-1] ≤ a[j]` (vì `j ≥ i+k-1`). Vậy **cửa sổ liền kề bắt đầu tại i** không tệ hơn.

→ Chỉ cần xét **n − k + 1** cửa sổ: `a[i+k-1] − a[i]`.

In [9]:
a = sorted([9, 4, 1, 7])
k = 3
print("Mọi nhóm 3 người trên mảng đã sort:")
for g in combinations(a, k):
    tag = "liền kề" if any(list(g) == a[i:i+k] for i in range(len(a) - k + 1)) else "có lỗ hổng"
    print(f"  {g}: hiệu = {max(g) - min(g)}  ({tag})")

Mọi nhóm 3 người trên mảng đã sort:
  (1, 4, 7): hiệu = 6  (liền kề)
  (1, 4, 9): hiệu = 8  (có lỗ hổng)
  (1, 7, 9): hiệu = 8  (có lỗ hổng)
  (4, 7, 9): hiệu = 5  (liền kề)


## 5. Cách 2 — Sort + cửa sổ trượt kích thước k (lời giải chính)

Giảng từng đoạn:

```python
nums.sort()                          # (1) sắp xếp tăng dần
best = float('inf')                  # (2) chưa có đáp án
for i in range(len(nums) - k + 1):   # (3) mỗi cửa sổ bắt đầu ở i
    diff = nums[i + k - 1] - nums[i] #     cuối cửa sổ − đầu cửa sổ
    best = min(best, diff)           # (4) giữ hiệu nhỏ nhất
return best
```

Vì đã sort, ta **không cần** gọi `max()`/`min()` trên cửa sổ: đầu là nhỏ nhất, cuối là lớn nhất → mỗi cửa sổ chỉ tốn `O(1)`.

Trường hợp `k = 1`: `nums[i] − nums[i] = 0` → đáp án 0, tự động đúng mà không cần `if`.

In [10]:
def min_diff_window(nums, k):
    a = sorted(nums)          # không phá mảng gốc của người gọi
    best = float('inf')
    for i in range(len(a) - k + 1):
        best = min(best, a[i + k - 1] - a[i])
    return best

print(min_diff_window([90], 1))           # 0
print(min_diff_window([9, 4, 1, 7], 2))   # 2
print(min_diff_window([9, 4, 1, 7], 3))   # 5
print(min_diff_window([87063, 61094, 44530, 21297, 95857, 93551, 9918], 6))

0
2
5
74560


## 6. TRACE từng cửa sổ

Ví dụ: `nums = [10, 1, 7, 3, 12, 5]`, `k = 3`.

Sau sort: `a = [1, 3, 5, 7, 10, 12]`, `n = 6` → có `6 − 3 + 1 = 4` cửa sổ.

| i | cửa sổ | đầu `a[i]` | cuối `a[i+2]` | diff | best sau |
|:-:|---|:-:|:-:|:-:|:-:|
| 0 | [1, 3, 5] | 1 | 5 | 4 | 4 |
| 1 | [3, 5, 7] | 3 | 7 | 4 | 4 |
| 2 | [5, 7, 10] | 5 | 10 | 5 | 4 |
| 3 | [7, 10, 12] | 7 | 12 | 5 | 4 |

Đáp án: **4**.

In [11]:
def trace_min_diff(nums, k):
    a = sorted(nums)
    print(f"nums = {nums}, k = {k}")
    print(f"sau sort: a = {a}")
    best = float('inf')
    for i in range(len(a) - k + 1):
        diff = a[i + k - 1] - a[i]
        old = best
        best = min(best, diff)
        note = "cập nhật" if best < old else "giữ nguyên"
        print(f"  i={i}: cửa sổ {a[i:i+k]} | {a[i+k-1]} - {a[i]} = {diff} | best: {old} -> {best} ({note})")
    print("Đáp án:", best)
    return best

trace_min_diff([10, 1, 7, 3, 12, 5], 3)
print()
trace_min_diff([9, 4, 1, 7], 2)

nums = [10, 1, 7, 3, 12, 5], k = 3
sau sort: a = [1, 3, 5, 7, 10, 12]
  i=0: cửa sổ [1, 3, 5] | 5 - 1 = 4 | best: inf -> 4 (cập nhật)
  i=1: cửa sổ [3, 5, 7] | 7 - 3 = 4 | best: 4 -> 4 (giữ nguyên)
  i=2: cửa sổ [5, 7, 10] | 10 - 5 = 5 | best: 4 -> 4 (giữ nguyên)
  i=3: cửa sổ [7, 10, 12] | 12 - 7 = 5 | best: 4 -> 4 (giữ nguyên)
Đáp án: 4

nums = [9, 4, 1, 7], k = 2
sau sort: a = [1, 4, 7, 9]
  i=0: cửa sổ [1, 4] | 4 - 1 = 3 | best: inf -> 3 (cập nhật)
  i=1: cửa sổ [4, 7] | 7 - 4 = 3 | best: 3 -> 3 (giữ nguyên)
  i=2: cửa sổ [7, 9] | 9 - 7 = 2 | best: 3 -> 2 (cập nhật)
Đáp án: 2


2

Nhìn trace ví dụ 2 (`[9,4,1,7]`, `k=2`): cặp tốt nhất là `(7, 9)` — trong mảng gốc chúng ở vị trí 0 và 3, **không** liền kề. Chỉ sau khi sort chúng mới đứng cạnh nhau. Đây là lý do bắt buộc phải sort.

## 7. Code nộp LeetCode

Trên LeetCode được phép sort tại chỗ `nums.sort()` (tiết kiệm bộ nhớ hơn `sorted`).

In [12]:
from typing import List

class Solution:
    def minimumDifference(self, nums: List[int], k: int) -> int:
        nums.sort()
        best = float('inf')
        for i in range(len(nums) - k + 1):
            best = min(best, nums[i + k - 1] - nums[i])
        return best

for nums, k, expected in [([90], 1, 0), ([9, 4, 1, 7], 2, 2)]:
    got = Solution().minimumDifference(list(nums), k)
    print(nums, k, "->", got)
    assert got == expected

[90] 1 -> 0
[9, 4, 1, 7] 2 -> 2


## 8. Cách 3 — Viết gọn bằng `zip` và `min` (Pythonic)

`zip(a, a[k-1:])` ghép cặp `(a[0], a[k-1])`, `(a[1], a[k])`, ... — đúng là cặp (đầu, cuối) của từng cửa sổ. `zip` dừng ở list ngắn hơn nên tự có đúng `n − k + 1` cặp.

Biểu thức `min(hi - lo for lo, hi in ...)` là **generator expression**: tính từng giá trị khi cần, không tạo list trung gian.

In [13]:
a = [1, 3, 5, 7, 10, 12]
k = 3
print("các cặp (đầu, cuối):", list(zip(a, a[k - 1:])))

def min_diff_zip(nums, k):
    a = sorted(nums)
    return min(hi - lo for lo, hi in zip(a, a[k - 1:]))

print(min_diff_zip([10, 1, 7, 3, 12, 5], 3))
print(min_diff_zip([90], 1))

các cặp (đầu, cuối): [(1, 5), (3, 7), (5, 10), (7, 12)]
4
0


### Code nộp LeetCode — Cách 3

In [14]:
class SolutionZip:
    def minimumDifference(self, nums: List[int], k: int) -> int:
        nums.sort()
        return min(hi - lo for lo, hi in zip(nums, nums[k - 1:]))

print(SolutionZip().minimumDifference([9, 4, 1, 7], 2))

2


## 9. Cách 4 — Hai con trỏ `left`, `right` (cửa sổ trượt "chuẩn")

Đây là cùng thuật toán nhưng viết theo khuôn **sliding window** tổng quát: `right` tiến từng bước, khi cửa sổ đủ `k` phần tử thì tính, rồi `left` tiến. Khuôn này sẽ dùng lại ở các bài cửa sổ trượt khác (tổng cửa sổ, đếm phần tử trong cửa sổ, ...).

In [15]:
def min_diff_two_pointers(nums, k):
    a = sorted(nums)
    best = float('inf')
    left = 0
    for right in range(len(a)):
        if right - left + 1 == k:       # cửa sổ [left..right] đủ k phần tử
            best = min(best, a[right] - a[left])
            left += 1                   # trượt cửa sổ sang phải
    return best

print(min_diff_two_pointers([10, 1, 7, 3, 12, 5], 3))
print(min_diff_two_pointers([9, 4, 1, 7], 2))
print(min_diff_two_pointers([90], 1))

4
2
0


## 10. Vì sao đúng? (lập luận thay thế)

**Khẳng định:** sau khi sort thành `a`, tồn tại một nhóm tối ưu gồm `k` phần tử **liên tiếp**.

**Chứng minh:** lấy nhóm tối ưu bất kỳ, gọi chỉ số (trong `a`) của phần tử nhỏ nhất là `i`, lớn nhất là `j`. Nhóm có `k` phần tử đều nằm trong `a[i..j]` nên `j − i + 1 ≥ k`, tức `j ≥ i + k − 1`.

Xét cửa sổ `a[i..i+k-1]`: nó có đúng `k` phần tử, min là `a[i]`, max là `a[i+k-1] ≤ a[j]` (mảng tăng). Vậy hiệu của cửa sổ `≤ a[j] − a[i]` = hiệu nhóm tối ưu → cửa sổ cũng tối ưu.

**Invariant vòng lặp:** sau khi xét cửa sổ `i`, `best` = hiệu nhỏ nhất trong các cửa sổ `0..i`. Duyệt hết `n − k + 1` cửa sổ → `best` là đáp án.

## 11. Time / Space và bảng so sánh

Gọi `n = len(nums)`.

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Vét cạn | thử mọi nhóm | $O(\binom{n}{k}\cdot k)$ | `O(k)` | Chỉ để kiểm chứng với n nhỏ |
| 2. Sort + cửa sổ (index) | `a[i+k-1] - a[i]` | `O(n log n)` | `O(n)` với `sorted`, `O(1)` thêm nếu sort tại chỗ* | **Khuyến nghị** |
| 3. Sort + `zip` | generator | `O(n log n)` | `O(n)` (slice `a[k-1:]`) | Ngắn gọn |
| 4. Hai con trỏ | khuôn sliding window | `O(n log n)` | như cách 2 | Dễ tổng quát hóa |

\*Timsort của Python có thể dùng tới `O(n)` bộ nhớ phụ khi sort; phần thuật toán sau sort chỉ `O(1)`.

Bước sort `O(n log n)` chiếm ưu thế; phần duyệt cửa sổ chỉ `O(n)`.

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

Các nhóm test:
1. Ví dụ đề: `[90], 1`; `[9,4,1,7], 2`.
2. Biên: `k = 1` (luôn 0), `k = n` (max − min toàn mảng).
3. Phần tử trùng: `[5,5,5,1], 3` → 0.
4. Phản ví dụ cho "không sort": `[9,4,1,7], 2` (cặp tốt nhất không liền kề trong mảng gốc).
5. Giá trị 0 và `10^5`.
6. Đối chiếu ngẫu nhiên (có seed) với vét cạn.

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<30} PASS ({actual})")

check("đề 1", min_diff_window([90], 1), 0)
check("đề 2", min_diff_window([9, 4, 1, 7], 2), 2)
check("k = 1", min_diff_window([3, 8, 1], 1), 0)
check("k = n", min_diff_window([3, 8, 1], 3), 7)
check("phần tử trùng", min_diff_window([5, 5, 5, 1], 3), 0)
check("giá trị biên 0 / 1e5", min_diff_window([0, 100000], 2), 100000)
check("cặp gần nhất ở cuối", min_diff_window([1, 10, 20, 21], 2), 1)

đề 1                           PASS (0)
đề 2                           PASS (2)
k = 1                          PASS (0)
k = n                          PASS (7)
phần tử trùng                  PASS (0)
giá trị biên 0 / 1e5           PASS (100000)
cặp gần nhất ở cuối            PASS (1)


In [17]:
import random

rng = random.Random(1984)
for _ in range(1500):
    n = rng.randint(1, 8)
    nums = [rng.randint(0, 30) for _ in range(n)]
    k = rng.randint(1, n)
    expected = min_diff_bruteforce(nums, k)
    for f in (min_diff_window, min_diff_zip, min_diff_two_pointers):
        assert f(nums, k) == expected, (f.__name__, nums, k)
    assert Solution().minimumDifference(list(nums), k) == expected
print("1500 test ngẫu nhiên: 4 cách khớp với vét cạn")

1500 test ngẫu nhiên: 4 cách khớp với vét cạn


## 13. Lỗi người mới hay gặp

### Lỗi 1: Quên sort, chỉ xét k phần tử liền kề trong mảng gốc

In [18]:
def wrong_no_sort(nums, k):
    best = float('inf')
    for i in range(len(nums) - k + 1):
        w = nums[i:i + k]
        best = min(best, max(w) - min(w))
    return best

print("sai :", wrong_no_sort([9, 4, 1, 7], 2))   # 3
print("đúng:", min_diff_window([9, 4, 1, 7], 2)) # 2

sai : 3
đúng: 2


**Vì sao sai:** đề cho chọn **bất kỳ** k người. Cặp `(9, 7)` không liền kề trong mảng gốc nên bị bỏ sót. **Sửa:** sort trước.

### Lỗi 2: Sai biên vòng lặp — `range(len(nums) - k)`

In [19]:
def wrong_range(nums, k):
    a = sorted(nums)
    best = float('inf')
    for i in range(len(a) - k):          # thiếu +1
        best = min(best, a[i + k - 1] - a[i])
    return best

print("sai (k = n):", wrong_range([3, 8, 1], 3))   # inf — không có cửa sổ nào!
print("sai (bỏ sót cửa sổ cuối):", wrong_range([1, 10, 20, 21], 2))  # 9, đúng là 1
print("đúng:", min_diff_window([3, 8, 1], 3), min_diff_window([1, 10, 20, 21], 2))

sai (k = n): inf
sai (bỏ sót cửa sổ cuối): 9
đúng: 7 1


**Vì sao sai:** cửa sổ cuối bắt đầu ở `n − k`, nên `range` phải tới `n − k + 1`. Khi `k = n` thì vòng rỗng và trả về `inf`. **Sửa:** `range(len(a) - k + 1)`.

### Lỗi 3: Dùng `a[i + k] - a[i]` (cửa sổ k + 1 phần tử)

In [20]:
def wrong_off_by_one(nums, k):
    a = sorted(nums)
    return min(a[i + k] - a[i] for i in range(len(a) - k))

print("sai :", wrong_off_by_one([10, 1, 7, 3, 12, 5], 3))   # xét cửa sổ 4 phần tử
print("đúng:", min_diff_window([10, 1, 7, 3, 12, 5], 3))

sai : 6
đúng: 4


**Vì sao sai:** từ `i` tới `i + k` là `k + 1` phần tử. Phần tử cuối của cửa sổ k phần tử là `i + k − 1`. Mẹo nhớ: **"đếm cả hai đầu: cuối − đầu + 1 = k"**.

### Lỗi 4: `best = 0` khi khởi tạo
`min(0, diff)` luôn là 0 → luôn trả 0. **Sửa:** khởi tạo `float('inf')` hoặc bằng hiệu của cửa sổ đầu tiên.

In [21]:
best = 0
for diff in [4, 4, 5, 5]:
    best = min(best, diff)
print("khởi tạo 0 -> best =", best, "(sai, đúng là 4)")

khởi tạo 0 -> best = 0 (sai, đúng là 4)


## 14. Mẹo nhận dạng pattern

- "Chọn **k** phần tử bất kỳ" + mục tiêu chỉ phụ thuộc **max và min** (hay "độ chênh", "khoảng") → **sort**, rồi phần tử tối ưu thường **liền kề**.
- "Cửa sổ kích thước cố định k" → `range(n - k + 1)`, cuối cửa sổ là `i + k - 1`.
- Khi thấy vét cạn `C(n, k)` → hỏi: thứ tự gốc có quan trọng không? Nếu không → sort để cấu trúc hóa không gian tìm kiếm.

## 15. Biến thể

1. **Trả về chính nhóm k người** tốt nhất (không chỉ hiệu).
2. **LeetCode 1200 — Minimum Absolute Difference:** `k = 2`, liệt kê mọi cặp có hiệu nhỏ nhất.
3. **Lớn nhất thay vì nhỏ nhất:** hiệu lớn nhất khi chọn k người = `max(nums) − min(nums)` (với `k ≥ 2`) — không cần cửa sổ.
4. **Đếm số cửa sổ** đạt hiệu tối ưu.

In [22]:
def best_group(nums, k):
    a = sorted(nums)
    best_i = min(range(len(a) - k + 1), key=lambda i: a[i + k - 1] - a[i])
    return a[best_i:best_i + k]

def all_min_pairs(arr):
    a = sorted(arr)
    d = min(y - x for x, y in zip(a, a[1:]))
    return [[x, y] for x, y in zip(a, a[1:]) if y - x == d]

print("nhóm tốt nhất:", best_group([10, 1, 7, 3, 12, 5], 3))
print("LC 1200:", all_min_pairs([4, 2, 1, 3]))

nhóm tốt nhất: [1, 3, 5]
LC 1200: [[1, 2], [2, 3], [3, 4]]


Giải thích `min(range(...), key=...)`: tham số `key` cho `min` biết "so sánh theo tiêu chí nào". Ở đây `min` trả về **chỉ số `i`** có hiệu cửa sổ nhỏ nhất (nếu hòa thì lấy `i` nhỏ nhất).

## 16. Bài tập tự luyện (TODO — chưa chạy)

### Bài 1 — Tính tay
`nums = [6, 2, 11, 9, 3]`, `k = 3`. Sort, liệt kê mọi cửa sổ và hiệu, cho đáp án.

### Bài 2 — Dự đoán output
`wrong_no_sort([1, 100, 2], 2)` trả về bao nhiêu? Đáp án đúng là bao nhiêu?

### Bài 3 — Tự viết lời giải chính

In [23]:
def exercise_min_diff(nums: list[int], k: int) -> int:
    # TODO 1: sort
    # TODO 2: duyệt i trong range(n - k + 1)
    # TODO 3: cập nhật best với a[i + k - 1] - a[i]
    pass

# assert exercise_min_diff([9, 4, 1, 7], 2) == 2
# assert exercise_min_diff([90], 1) == 0

### Bài 4 — Biến thể: trả về số cửa sổ (sau sort) đạt hiệu nhỏ nhất

In [24]:
def exercise_count_best_windows(nums: list[int], k: int) -> int:
    # TODO: tính hiệu mọi cửa sổ, đếm số lần bằng hiệu nhỏ nhất
    pass

# assert exercise_count_best_windows([10, 1, 7, 3, 12, 5], 3) == 2
# assert exercise_count_best_windows([1, 2, 3, 4], 2) == 3

### Bài 5 — Biến thể: tổng lớn nhất của k phần tử **liên tiếp trong mảng gốc** (không sort) — cửa sổ trượt cộng/trừ

In [25]:
def exercise_max_window_sum(nums: list[int], k: int) -> int:
    # TODO: tính tổng cửa sổ đầu, rồi mỗi bước + phần tử mới - phần tử rời cửa sổ
    pass

# assert exercise_max_window_sum([1, 12, -5, -6, 50, 3], 4) == 51
# assert exercise_max_window_sum([5], 1) == 5

### Bài 6 — Mini challenge: chọn k người sao cho hiệu ≤ `limit`; trả về **k lớn nhất** có thể

In [26]:
def exercise_max_group_within(nums: list[int], limit: int) -> int:
    # TODO: sort + hai con trỏ; cửa sổ hợp lệ khi a[right] - a[left] <= limit
    pass

# assert exercise_max_group_within([1, 3, 6, 7, 9], 3) == 3   # {6, 7, 9}
# assert exercise_max_group_within([5], 0) == 1

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Sort trước | `[2, 3, 6, 9, 11]`, 3 cửa sổ | hiệu 4, 6, 5 |
| 2 | Không sort chỉ xét cặp kề | cặp (1,100), (100,2) | sau sort: (1,2) |
| 3 | Khuôn mục 5 | `best = float('inf')` | `a[i + k - 1] - a[i]` |
| 4 | Lưu list các hiệu | `d = min(diffs)` | `diffs.count(d)` |
| 5 | Tổng cửa sổ đầu = `sum(nums[:k])` | `s += nums[i] - nums[i-k]` | cập nhật `best = max(best, s)` |
| 6 | Sort, cửa sổ co giãn | `while a[right] - a[left] > limit: left += 1` | `best = max(best, right - left + 1)` |

## 18. Đáp án tham khảo, template và checklist

- Bài 1: sort → `[2, 3, 6, 9, 11]`; cửa sổ `[2,3,6]`→4, `[3,6,9]`→6, `[6,9,11]`→5 → đáp án **4**.
- Bài 2: `wrong_no_sort([1, 100, 2], 2)` = **98** (cặp kề (1,100)=99, (100,2)=98); đáp án đúng = **1** (cặp 1, 2).

In [27]:
def answer_min_diff(nums, k):
    a = sorted(nums)
    best = float('inf')
    for i in range(len(a) - k + 1):
        best = min(best, a[i + k - 1] - a[i])
    return best

def answer_count_best_windows(nums, k):
    a = sorted(nums)
    diffs = [a[i + k - 1] - a[i] for i in range(len(a) - k + 1)]
    return diffs.count(min(diffs))

def answer_max_window_sum(nums, k):
    s = sum(nums[:k])
    best = s
    for i in range(k, len(nums)):
        s += nums[i] - nums[i - k]
        best = max(best, s)
    return best

def answer_max_group_within(nums, limit):
    a = sorted(nums)
    best = left = 0
    for right in range(len(a)):
        while a[right] - a[left] > limit:
            left += 1
        best = max(best, right - left + 1)
    return best

assert answer_min_diff([6, 2, 11, 9, 3], 3) == 4
assert wrong_no_sort([1, 100, 2], 2) == 98 and answer_min_diff([1, 100, 2], 2) == 1
assert answer_min_diff([9, 4, 1, 7], 2) == 2 and answer_min_diff([90], 1) == 0
assert answer_count_best_windows([10, 1, 7, 3, 12, 5], 3) == 2
assert answer_count_best_windows([1, 2, 3, 4], 2) == 3
assert answer_max_window_sum([1, 12, -5, -6, 50, 3], 4) == 51
assert answer_max_window_sum([5], 1) == 5
assert answer_max_group_within([1, 3, 6, 7, 9], 3) == 3
assert answer_max_group_within([5], 0) == 1
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


### Template nhớ nhanh

```python
class Solution:
    def minimumDifference(self, nums: List[int], k: int) -> int:
        nums.sort()
        return min(nums[i + k - 1] - nums[i] for i in range(len(nums) - k + 1))
```

Câu thần chú: **"Sort → k phần tử liền kề → cuối trừ đầu."**

### Checklist trước khi submit
- [ ] Đã **sort** trước khi xét cửa sổ?
- [ ] Vòng lặp là `range(n - k + 1)` (có `+1`)?
- [ ] Cuối cửa sổ là `i + k - 1` (không phải `i + k`)?
- [ ] `best` khởi tạo `inf`, không phải 0?
- [ ] Đã test `k = 1` (đáp án 0) và `k = n`?